In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("data/cdss.db")

print("Loading side effects...")
se_df = pd.read_csv("data/raw/drug_side_effect_with_names.csv")
se_df.columns = [c.lower() for c in se_df.columns]
se_df = se_df[['drug_id', 'side_effect_name']]
se_df.to_sql("side_effects", conn, if_exists="replace", index=False)

print("Loading diseases...")
dz_df = pd.read_csv("data/raw/drug_disease_with_names.csv")
dz_df.columns = [c.lower() for c in dz_df.columns]
dz_df = dz_df[['drug_id', 'disease_name']]
dz_df.to_sql("diseases", conn, if_exists="replace", index=False)

print("Creating indexes...")
conn.execute("CREATE INDEX IF NOT EXISTS idx_se_drug ON side_effects(drug_id)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_dz_drug ON diseases(drug_id)")
conn.commit()

print("✅ Side effects and diseases loaded into database!")

Loading side effects...
Loading diseases...


ParserError: Error tokenizing data. C error: Expected 4 fields in line 4838, saw 7


In [2]:
def search_drug(query):
    sql = """
    SELECT display_name, half_life, indication, toxicity 
    FROM drug_dictionary 
    WHERE search_lower LIKE ? 
    LIMIT 1
    """
    df = pd.read_sql_query(sql, conn, params=(f"%{query.lower()}%",))
    if df.empty:
        return f"No drug found for '{query}'."
    
    row = df.iloc[0]
    print(f"💊 {row['display_name']}")
    print(f"⏳ Half-Life: {row['half_life']}")
    print(f"🎯 Indication: {row['indication'][:200]}...")
    print(f"⚠️ Toxicity: {row['toxicity'][:200]}...")

search_drug("Amiodarone")

💊 Amiodarone
⏳ Half-Life: N/A
🎯 Indication: N/A...
⚠️ Toxicity: N/A...


In [3]:
def check_ddi(drug1, drug2):
    sql = """
    SELECT drug1_name, drug2_name, severity, mechanism
    FROM ddi_rules
    WHERE (LOWER(drug1_name) = LOWER(?) AND LOWER(drug2_name) = LOWER(?))
       OR (LOWER(drug1_name) = LOWER(?) AND LOWER(drug2_name) = LOWER(?))
    LIMIT 5
    """
    df = pd.read_sql_query(sql, conn, params=(drug1, drug2, drug2, drug1))
    if df.empty:
        print(f"✅ No major interaction found between {drug1} and {drug2}.")
    else:
        print(f"🚨 Interactions found:")
        display(df)

check_ddi("Warfarin", "Aspirin")

✅ No major interaction found between Warfarin and Aspirin.


In [4]:
def analyze_patient_profile(meds_list):
    print(f"🩺 Analyzing regimen: {', '.join(meds_list)}\n")
    
    found_interactions = []
    
    # Check every possible pair of drugs
    for drug_a, drug_b in combinations(meds_list, 2):
        sql = """
        SELECT drug1_name, drug2_name, severity, mechanism
        FROM ddi_rules
        WHERE (LOWER(drug1_name) = LOWER(?) AND LOWER(drug2_name) = LOWER(?))
        LIMIT 1
        """
        df = pd.read_sql_query(sql, conn, params=(drug_a, drug_b))
        if not df.empty:
            found_interactions.append(df.iloc[0])
            
    if not found_interactions:
        print("✅ No known interactions between these medications.")
    else:
        print(f"⚠️ Found {len(found_interactions)} potential interaction(s):\n")
        
        for i, ddi in enumerate(found_interactions, 1):
            sev = ddi['severity']
            mech = ddi['mechanism']
            print(f"{i}. {ddi['drug1_name']} + {ddi['drug2_name']}  [Severity: {sev}]")
            print(f"   Mechanism: {mech[:150]}...\n")

# Change this list to test different patients!
patient_meds = ["Warfarin", "Amiodarone", "Simvastatin", "Aspirin"]
analyze_patient_profile(patient_meds)

🩺 Analyzing regimen: Warfarin, Amiodarone, Simvastatin, Aspirin

⚠️ Found 3 potential interaction(s):

1. Warfarin + Amiodarone  [Severity: Moderate]
   Mechanism: The risk or severity of bleeding can be increased when Amiodarone is combined with Warfarin....

2. Warfarin + Simvastatin  [Severity: Minor]
   Mechanism: Simvastatin may increase the anticoagulant activities of Warfarin....

3. Amiodarone + Simvastatin  [Severity: Moderate]
   Mechanism: The serum concentration of Simvastatin can be increased when it is combined with Amiodarone....

